# Phase 1: diverse grounded legal RAG (Llama-3.2-1B + latent module)

Before running: Settings -> **Accelerator: GPU T4 x2**, **Internet: ON**, and add a Kaggle **Secret** named `HF_TOKEN` (Add-ons -> Secrets).
Run the cells top to bottom. Steps 3 and 4 take the longest, so run them as separate cells.

## 0. Setup

In [ ]:
import os
REPO = 'https://github.com/ardrarachel/llm-project.git'
if not os.path.exists('/kaggle/working/llm-project'):
    !git clone {REPO} /kaggle/working/llm-project
%cd /kaggle/working/llm-project
!git pull
!pip install -q faiss-cpu sentence-transformers nltk pyyaml

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import torch, transformers
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('torch', torch.__version__, '| transformers', transformers.__version__)

## 1. Get the data
Tries the Dropbox link. If it fails, download the zip in your browser, upload it as a Kaggle Dataset, add it to this notebook, and skip this cell (the config already searches /kaggle/input).

In [ ]:
!python notebooks/00_download_data.py

## 2. Chunk, index, retrieve + measure retrieval quality

In [ ]:
!python scripts/01_prepare_data.py

Write the precision / recall numbers in your diary. Low precision is normal: we retrieve 3 chunks of 512 characters for a short gold clause.

## 3. Build the self-supervised training data (Llama writes 4 answers per question)

In [ ]:
!python scripts/02_generate_training_data.py

In [ ]:
import json
rows=[json.loads(l) for l in open('data/processed/train_responses.jsonl')]
print(len(rows),'training groups')
r=rows[0]; print(r['query'],'\n')
for s,t in zip(r['styles'],r['responses']): print(f'[{s}]',t,'\n')

**Read these answers.** If they are empty, repeat the question, or ignore the excerpts, fix the data before training.

## 4. Train the latent module (backbone frozen)
If Kaggle stops the session, re-run this cell with `--resume`.

In [ ]:
!python scripts/03_train.py

Check the log: **KL(inst)** should rise above ~1 and validation `active_units` should be > 0. KL near 0 means posterior collapse (see the troubleshooting notes).

## 5. Evaluate: latent sampling vs plain RAG baseline (test contracts)

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
import json
s=json.load(open('outputs/eval_samples.json'))
for x in s[:3]:
    print('Q:',x['query'],'\n')
    print('--- baseline ---');  [print('*',t,'\n') for t in x['baseline']]
    print('--- latent (4 different z) ---'); [print('*',t,'\n') for t in x['latent']]
    print('='*80)

## 6. Save your work
Commit the small result files (not the checkpoints) so they survive the session:

In [ ]:
!git config user.email 'you@example.com' && git config user.name 'Ardra'
!git add outputs/*.json data/processed/retrieval_metrics.json
!git commit -m 'Phase 1 results' || true
# push needs a GitHub token: git remote set-url origin https://<user>:<token>@github.com/ardrarachel/llm-project.git ; git push
# Keep the model: click 'Save Version' on Kaggle, or upload outputs/phase1_latent.pt to your Hugging Face account.